In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 15 · Migration Path to Foundry

Most teams already have working code against the OpenAI API directly (or another cloud). The fear is a rewrite. The reality is a **four-line diff**: same `openai` SDK, same chat-completions shape — you swap the client constructor for `AzureOpenAI` with managed-identity auth and keep everything else. *Move to Foundry by Friday, not by Q3.*


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every model call below shows up live in the Microsoft Foundry portal under **your project → Tracing**.*


In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='sutter-live-demo')


---
## Step 1 — The 'before': OpenAI-direct (reference only)

This is the typical starting point. We **don't run** it — it needs an `OPENAI_API_KEY` and sends data to a third party. It's here to diff against.


In [ ]:
BEFORE = '''
from openai import OpenAI
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])   # 3rd-party, API key
resp = client.chat.completions.create(
    model='gpt-4o',
    messages=[{'role': 'user', 'content': prompt}])
print(resp.choices[0].message.content)
'''
print(BEFORE)


---
## Step 2 — The 'after': Azure Foundry (LIVE)

Same SDK family, same call. The only changes: `AzureOpenAI` constructor, your endpoint, an API version, and a **managed-identity token provider** (no keys). Run it for real.


In [ ]:
from openai import AzureOpenAI
from azure.identity import get_bearer_token_provider
from _advisor import get_credential

token_provider = get_bearer_token_provider(
    get_credential(), 'https://cognitiveservices.azure.com/.default')

client = AzureOpenAI(
    azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
    api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
    azure_ad_token_provider=token_provider)          # <- no API key

prompt = 'In one sentence, what is Sutter Health Member Services?'
resp = client.chat.completions.create(
    model=os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini'),  # deployment name
    max_tokens=80,
    messages=[{'role': 'user', 'content': prompt}])
print('LIVE Foundry answer:', resp.choices[0].message.content)


---
## Step 3 — The diff, line by line

Exactly what changed between *before* and *after* — and nothing else in your business logic moves.


In [ ]:
DIFF = '''
- from openai import OpenAI
+ from openai import AzureOpenAI
+ from azure.identity import get_bearer_token_provider, DefaultAzureCredential

- client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
+ client = AzureOpenAI(
+     azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
+     api_version='2025-04-01-preview',
+     azure_ad_token_provider=get_bearer_token_provider(
+         DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default'))

  resp = client.chat.completions.create(   # UNCHANGED
-     model='gpt-4o',
+     model='<your-deployment-name>',       # deployment, not model id
      messages=[{'role': 'user', 'content': prompt}])
'''
print(DIFF)
print('Net change: swap constructor + auth + model->deployment. Logic untouched.')


---
## Takeaways

- Migration is a **constructor + auth swap**, not a rewrite — the `chat.completions` surface is identical.
- You gain **managed identity (no keys)**, data residency, private networking and fine-tuning.
- From other clouds (Bedrock/Vertex), the same pattern applies: keep your orchestration, point the client at Foundry.
- Once ported, Labs 01-14 (fine-tune, route, govern, secure) are all available.

*← The Decision Advisor (Lab 09) routes the `needs_migration` flag here.*
